In [16]:
!pip install polars duckdb pyarrow plotly

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
import polars as pl
import duckdb

DATA_PATH = "../data/raw/transjakarta_gps.csv"

# Tambahkan ignore_errors=True
lazy_df = pl.scan_csv(DATA_PATH, ignore_errors=True)

# Tampilkan 5 baris pertama
df_head = lazy_df.limit(5).collect()
print(df_head)

shape: (5, 11)
┌──────────┬─────────┬──────────────┬──────────┬───┬──────────┬───────┬────────┬──────────────┐
│ bus_code ┆ trip_id ┆ gps_datetime ┆ location ┆ … ┆ latitude ┆ speed ┆ course ┆ color        │
│ ---      ┆ ---     ┆ ---          ┆ ---      ┆   ┆ ---      ┆ ---   ┆ ---    ┆ ---          │
│ str      ┆ f64     ┆ str          ┆ str      ┆   ┆ f64      ┆ i64   ┆ i64    ┆ str          │
╞══════════╪═════════╪══════════════╪══════════╪═══╪══════════╪═══════╪════════╪══════════════╡
│ DMR 5049 ┆ 5.112   ┆ 2019-11-26   ┆          ┆ … ┆ -6.22423 ┆ 0     ┆ 208    ┆ MERAH KUNING │
│          ┆         ┆ 14:00:29     ┆          ┆   ┆          ┆       ┆        ┆              │
│ DMR 5049 ┆ 5.112   ┆ 2019-11-26   ┆          ┆ … ┆ -6.22423 ┆ 0     ┆ 208    ┆ MERAH KUNING │
│          ┆         ┆ 14:00:54     ┆          ┆   ┆          ┆       ┆        ┆              │
│ DMR 5049 ┆ 5.112   ┆ 2019-11-26   ┆          ┆ … ┆ -6.22423 ┆ 0     ┆ 208    ┆ MERAH KUNING │
│          ┆         ┆ 14

In [18]:
# Menghitung total baris dan kolom
total_rows = lazy_df.select(pl.len()).collect().item()
schema = lazy_df.collect_schema()

print(f"Total Baris Data: {total_rows:,}")
print(f"Jumlah Kolom: {len(schema)}")
print("\nSkema Tipe Data:")
for col, dtype in schema.items():
    print(f"- {col}: {dtype}")

Total Baris Data: 775,458
Jumlah Kolom: 11

Skema Tipe Data:
- bus_code: String
- trip_id: Float64
- gps_datetime: String
- location: String
- dtd: Float64
- corridor: Int64
- longitude: Float64
- latitude: Float64
- speed: Int64
- course: Int64
- color: String


In [19]:
# Mengambil sampel 5 baris pertama
df_head = lazy_df.limit(5).collect()
print(df_head)

shape: (5, 11)
┌──────────┬─────────┬──────────────┬──────────┬───┬──────────┬───────┬────────┬──────────────┐
│ bus_code ┆ trip_id ┆ gps_datetime ┆ location ┆ … ┆ latitude ┆ speed ┆ course ┆ color        │
│ ---      ┆ ---     ┆ ---          ┆ ---      ┆   ┆ ---      ┆ ---   ┆ ---    ┆ ---          │
│ str      ┆ f64     ┆ str          ┆ str      ┆   ┆ f64      ┆ i64   ┆ i64    ┆ str          │
╞══════════╪═════════╪══════════════╪══════════╪═══╪══════════╪═══════╪════════╪══════════════╡
│ DMR 5049 ┆ 5.112   ┆ 2019-11-26   ┆          ┆ … ┆ -6.22423 ┆ 0     ┆ 208    ┆ MERAH KUNING │
│          ┆         ┆ 14:00:29     ┆          ┆   ┆          ┆       ┆        ┆              │
│ DMR 5049 ┆ 5.112   ┆ 2019-11-26   ┆          ┆ … ┆ -6.22423 ┆ 0     ┆ 208    ┆ MERAH KUNING │
│          ┆         ┆ 14:00:54     ┆          ┆   ┆          ┆       ┆        ┆              │
│ DMR 5049 ┆ 5.112   ┆ 2019-11-26   ┆          ┆ … ┆ -6.22423 ┆ 0     ┆ 208    ┆ MERAH KUNING │
│          ┆         ┆ 14

In [20]:
# Pengecekan Missing Values per kolom
missing_counts = lazy_df.select([
    pl.col(c).null_count().alias(c) for c in schema.keys()
]).collect()

print("Jumlah Nilai Kosong (Null):")
print(missing_counts)

Jumlah Nilai Kosong (Null):
shape: (1, 11)
┌──────────┬─────────┬──────────────┬──────────┬───┬──────────┬────────┬────────┬───────┐
│ bus_code ┆ trip_id ┆ gps_datetime ┆ location ┆ … ┆ latitude ┆ speed  ┆ course ┆ color │
│ ---      ┆ ---     ┆ ---          ┆ ---      ┆   ┆ ---      ┆ ---    ┆ ---    ┆ ---   │
│ u32      ┆ u32     ┆ u32          ┆ u32      ┆   ┆ u32      ┆ u32    ┆ u32    ┆ u32   │
╞══════════╪═════════╪══════════════╪══════════╪═══╪══════════╪════════╪════════╪═══════╡
│ 0        ┆ 499001  ┆ 0            ┆ 0        ┆ … ┆ 0        ┆ 314189 ┆ 0      ┆ 0     │
└──────────┴─────────┴──────────────┴──────────┴───┴──────────┴────────┴────────┴───────┘


In [21]:
# Profiling statistik deskriptif dengan DuckDB SUMMARIZE
con = duckdb.connect()
summary_df = con.execute(f"SUMMARIZE SELECT * FROM read_csv_auto('{DATA_PATH}')").df()

print(summary_df[['column_name', 'column_type', 'min', 'max', 'approx_unique', 'null_percentage']])

     column_name column_type                  min                  max  \
0       bus_code     VARCHAR             DMR 5049              TSW 100   
1        trip_id     VARCHAR                    -             TR1.3022   
2   gps_datetime   TIMESTAMP  2019-11-26 14:00:00  2019-11-26 17:59:59   
3       location     VARCHAR              1-1,1-2              9-9,9-8   
4            dtd      DOUBLE                  0.0                  1.0   
5       corridor     VARCHAR                    0                  T12   
6      longitude      DOUBLE              106.607           108.330973   
7       latitude      DOUBLE            -6.791528            -6.048802   
8          speed      BIGINT                    0                  309   
9         course      BIGINT                    0                  360   
10         color     VARCHAR                 BIRU           UNGU HIJAU   

    approx_unique  null_percentage  
0            1764             0.00  
1             759             0.00  
